# 09 — Final Model Persistence

## 1. Objective
The goal of this notebook is to execute **Phase 9 Final Model Persistence** by training the selected **Tuned Support Vector Machine** pipeline on the complete training dataset ($N=734$), saving the fitted `Pipeline` as a joblib binary artifact (`models/final_model.joblib`), creating project metadata (`models/model_metadata.json`), and verifying full prediction reproducibility on the held-out test set ($N=184$).

### Critical Methodological Rules:
1. **Training Data Only:** The final pipeline is trained strictly on `X_train` and `y_train` ($N=734$). The test set (`X_test`, `y_test`) is reserved for evaluation verification only.
2. **Complete Pipeline Serialization:** The saved artifact wraps the complete scikit-learn `Pipeline` (`preprocessor -> classifier`), ensuring raw-input inference compatibility.
3. **No Retraining on Test Data / No New Tuning:** The selected hyperparameters (`C=100, kernel='linear', gamma='scale', probability=True`) and preprocessing steps remain exact.


## 2. Load Prepared Model Data
Load modeling features and target, creating the 80/20 stratified train/test split.


In [1]:
import sys
from pathlib import Path

# Locate project root containing src/
root_dir = Path.cwd()
for p in [root_dir] + list(root_dir.parents):
    if (p / 'src' / 'config.py').exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

import pandas as pd
import numpy as np
import json
import hashlib
from IPython.display import display

from src.config import MODELS_DIR, REPORTS_DIR, RANDOM_STATE, DATASET_PATH
from src.data_loader import load_model_data, split_data
from src.model_persistence import (
    get_final_model,
    train_final_model,
    get_model_metadata,
    save_final_model,
    load_final_model,
    verify_persisted_model,
    run_model_persistence_workflow,
)

# Load data and prepare 80/20 train/test split
X, y = load_model_data()
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.20, random_state=RANDOM_STATE)

print(f"Training set: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Held-out test set: X_test = {X_test.shape}, y_test = {y_test.shape}")


Training set: X_train = (734, 10), y_train = (734,)
Held-out test set: X_test = (184, 10), y_test = (184,)


## 3. Train Selected Final Pipeline
Train the complete `Pipeline(steps=[('preprocessor', preprocessor), ('model', SVC(...))])` on `X_train`, `y_train`.


In [2]:
fitted_pipeline = train_final_model(X_train, y_train)
print("Successfully trained final selected pipeline on X_train!")
print("Pipeline steps:", fitted_pipeline.named_steps)


Successfully trained final selected pipeline on X_train!
Pipeline steps: {'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalch',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
         

## 4. Save Pipeline & Metadata Artifacts
Save `final_model.joblib` and `model_metadata.json` into `models/`.


In [3]:
metadata = get_model_metadata(X_train, test_sample_count=len(X_test))
model_path, metadata_path = save_final_model(fitted_pipeline, metadata, MODELS_DIR)

print(f"Model saved to: {model_path} ({model_path.stat().st_size} bytes)")
print(f"Metadata saved to: {metadata_path} ({metadata_path.stat().st_size} bytes)")


Model saved to: C:\projects\Disease-Diagnosis-Prediction\models\final_model.joblib (55830 bytes)
Metadata saved to: C:\projects\Disease-Diagnosis-Prediction\models\model_metadata.json (1119 bytes)


## 5. Load Persisted Artifacts
Deserialize `models/final_model.joblib` and `models/model_metadata.json` back into memory.


In [4]:
loaded_pipeline, loaded_metadata = load_final_model(MODELS_DIR)
print("Successfully loaded persisted pipeline!")
print("Metadata content:")
print(json.dumps(loaded_metadata, indent=2))


Successfully loaded persisted pipeline!
Metadata content:
{
  "project_name": "Disease Diagnosis Prediction",
  "model_name": "Tuned Support Vector Machine",
  "model_type": "SVC",
  "hyperparameters": {
    "C": 100,
    "kernel": "linear",
    "gamma": "scale",
    "probability": true,
    "random_state": 42
  },
  "preprocessing_description": "Median imputation for missing numericals (chol), standard scaling, and drop=None one-hot encoding for categorical variables.",
  "training_sample_count": 734,
  "test_sample_count": 184,
  "number_of_input_features": 10,
  "feature_names": [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak",
    "sex",
    "cp",
    "restecg",
    "fbs",
    "exang"
  ],
  "random_state": 42,
  "test_split_configuration": {
    "test_size": 0.2,
    "stratify": true,
    "random_state": 42
  },
  "model_selection_reference": "Phase 8 Final Model Selection",
  "creation_timestamp": "2026-09-15T11:13:01.059423+00:00",
  "sklearn_version": "1.6.1

## 6. Verify Pipeline Structure & Hyperparameters
Validate that the loaded object is a scikit-learn Pipeline with `preprocessor` and `model` steps and exact hyperparameters (`C=100, kernel='linear', probability=True`).


In [5]:
model_step = loaded_pipeline.named_steps["model"]
print(f"Model type: {type(model_step).__name__}")
print(f"  C: {model_step.C}")
print(f"  kernel: {model_step.kernel}")
print(f"  probability: {model_step.probability}")
print(f"  random_state: {model_step.random_state}")

assert model_step.C == 100
assert model_step.kernel == "linear"
assert model_step.probability is True
print("Hyperparameter verification passed!")


Model type: SVC
  C: 100
  kernel: linear
  probability: True
  random_state: 42
Hyperparameter verification passed!


## 7. Verify Prediction & Probability Reproducibility
Generate class predictions and probability scores on held-out test set (`X_test`) using the loaded pipeline and compare against the in-memory fitted pipeline outputs.


In [6]:
preds_orig = fitted_pipeline.predict(X_test)
probs_orig = fitted_pipeline.predict_proba(X_test)[:, 1]

preds_loaded = loaded_pipeline.predict(X_test)
probs_loaded = loaded_pipeline.predict_proba(X_test)[:, 1]

assert np.array_equal(preds_orig, preds_loaded), "Class predictions must match 100% exactly!"
assert np.allclose(probs_orig, probs_loaded, atol=1e-6), "Predicted probabilities must match 100% exactly!"

print("Prediction & probability reproducibility verification passed!")


Prediction & probability reproducibility verification passed!


## 8. Artifact Integrity & Raw Data Hash Verification
Verify `models/final_model.joblib` artifact integrity and verify raw CSV MD5 hash remains `13c9cfee54ce2b1552ef7d787a5d8be9` (918 observations).


In [7]:
raw_bytes = DATASET_PATH.read_bytes()
raw_md5 = hashlib.md5(raw_bytes).hexdigest()
raw_df = pd.read_csv(DATASET_PATH)

print(f"Raw CSV MD5: {raw_md5}")
print(f"Raw CSV Row Count: {len(raw_df)}")

assert raw_md5 == "13c9cfee54ce2b1552ef7d787a5d8be9"
assert len(raw_df) == 918
print("Raw data integrity verification passed!")


Raw CSV MD5: 13c9cfee54ce2b1552ef7d787a5d8be9
Raw CSV Row Count: 918
Raw data integrity verification passed!
